#### outline
---
談論不同的注意力機制，共實作四種:
1. 簡化的注意力機制(Simplified Attention): 先說明注意力機制的概念  
單純看相量和向量之間像不像，來理解**加權平均**的概念
2. 自注意力(Self-Attention): 加入可訓練的權重  
讓模型自己學會**應該要更關注誰才可以完成任務**
3. 因果注意力(Causal Attention): 關注**先前和現在**的輸入內容
4. 多頭注意力(Multi-Head Attention): 自注意力和因果注意力延伸的注意力機制，可以使模型注意到來自不同表徵子空間的資訊  
簡而言之就是分頭行動，有人看語法、有人看代名詞，等等

In [43]:
import torch

#### math
---
核心的數學概念:  
先定義參數:  
1. Query(Q): 目前在看的token，想看這個token和同序列中其他token的關係  
2. Key(K):  序列中的每個token拿來與query比對的特徵  
K專門用來和Q做點積(Dot Product)比對用的  
3. Value(V): 當這個token被選中後(Key)，實際上要貢獻出的資訊特徵

Scaled Dot-Product Attention:  
**$Attention(Q, K, V)= softmax(\frac{QK^{T}}{\sqrt{d_{k}}})V$**  
$\frac{1}{\sqrt{d_k}}$為縮放因子，防止點積結果過大導致Softmax梯度消失

### 3.3 利用自注意力機制，關注輸入序列中的不同位置
---
所謂的`self`是指model可以`自身`學習與評估輸入文本內容間每個token的關係和依賴程度，而機制的運作範圍侷限在`自身`的序列內部，所以可以注意到同一序列(同個句子)中的其他詞彙，並算出個別的注意力權重

![figure3.1](https://github.com/aqingo3o/1004-LLM/blob/d48c38ab1f22a76e9b0c0b5c6f9e5c2af53677cf/exp/RF/ch03/figure1.png)  
在自注意力中，透過**比較序列中每個元素間的資訊**，為序列中的每個元素($x^i$)**建立一個上下文向量($z^i$)**，所以**上下文向量是嵌入向量的升級版本**，裡面**包含了上下文關聯的資訊**  
引入一些專有動作，作為一個引路人:
1. `比較序列中每個元素間的資訊` $\rightarrow$ 點積($Query \cdot Key^T$)
2. `建立一個上下文向量(z^i)` $\rightarrow$ 加權求和 $z^i = \sum \alpha^{Ni} \cdot v^N $
3. `上下文向量是嵌入向量的升級版本` $\rightarrow$ 維度相同，代表同一個token
4. `包含了上下文關聯的資訊` $\rightarrow$ 最終目的

所以等會的實作環節會遵循以下三個步驟:
1. 比較資訊(Similarity):  
    $Score = \vec{Q} \cdot \vec{K}^T$
2. 決定權重(Normalization):  
    $\vec{\alpha} = Softmax(Score)$
3. 提取上下文(Context Aggregation):  
    $z = \sum(\vec{\alpha} \cdot \vec{v})$

##### 點積  
我不相信有任何人不知道點積的定義，所以下面只是show出拍透取的操作

In [44]:
vec_1 = torch.tensor([0.45, 0.12])
vec_2 = torch.tensor([0.64, 0.87])

res = 0.
for i, j in enumerate(vec_1):
    res += vec_1[i] * vec_2[i]

# 手動計算答案
print(res)

# pytorch函數計算(兩種方法)
print(torch.dot(vec_1, vec_2))
print(vec_1 @ vec_2)

tensor(0.3924)
tensor(0.3924)
tensor(0.3924)


#### 3.3.1 簡化得自注意力機制(不使用可訓練權重)

In [45]:
# 建立一個簡單序列
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],     # Your      (x^1)
        [0.55, 0.87, 0.66],     # journey   (x^2)
        [0.57, 0.85, 0.64],     # starts    (x^3)
        [0.22, 0.58, 0.33],     # with      (x^4)
        [0.77, 0.25, 0.10],     # one       (x^5)
        [0.05, 0.80, 0.55]      # step      (x^6)
    ]
)

In [46]:
# step1. 用點積的方式算出Score

query = inputs[1]   # 查詢目標: 假設我們要看的是第二個token
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)    # 讓所有元素和第二個元素做內積(Q \cdot K^T)
print(attn_scores_2)
# 內積完是純數值

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


所以輸出內容中的每一個元素，皆為原本序列中每個元素和第二個元素內積後的結果  
為什麼第二項不是1呀? 因為還沒normalized，長度不是1。

In [47]:
# step2. 將score做歸一化(v1. 用最簡單的*比例*)
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weight:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weight: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


有弊端，如果內積後為負數，負數的權重先不要吧

In [48]:
# step2. 將score做歸一化(v2. 用softmax)
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


還記得我們先前在[pytorch.ipynb](https://github.com/aqingo3o/1004-LLM/blob/d48c38ab1f22a76e9b0c0b5c6f9e5c2af53677cf/exp/RF/pytorch.ipynb)學到的使用`softmax`激活函數把原始分數壓成機率嗎?  
這邊做了一樣的事情，將score用softmax壓一壓，確保我們輸出的權重值永遠介在`0-1`之間

In [49]:
# step3. 計算最後的上下文向量
query = inputs[1]   # 一樣，我們只看第二個token
context_vec_2 = torch.zeros(query.shape)     # 建立一個空向量等等放數字進去
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i # 計算上下文向量(所有輸入向量的加權和) 
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


#### 3.3.2 計算所有輸入token的注意力權重
剛剛都只有關注第二個token，看他和別人的點積，進而壓成權重，最後變成上下文向量；現在要將序列中的每個token做一樣的事情

In [50]:
# step1. 算score
attn_scores = inputs @ inputs.T 
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


剛剛提到算score就是算點積，對於矩陣來說就是自己乘上自己的轉置(見fig.2)
![figure2](https://github.com/aqingo3o/1004-LLM/blob/40b5834b30949ce9f7eb4155ac6e737eeb9386ad/exp/RF/ch03/figure2.png)

In [51]:
# step2. normalized
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)
print("All row sums:", attn_weights.sum(dim=-1))    # 將每一列總和確認都是1

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


一樣使用softmax將權重壓成在0-1之間的數字，dim=-1代表橫向加總要為1

In [52]:
# step3. context aggregation
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


用fig.3建立圖像概念
![figure3](https://github.com/aqingo3o/1004-LLM/blob/5803ffabab688d730ef7c5365d579f911572403b/exp/RF/ch03/figure3.png)

### 3.4 使用`可訓練權重`實作自注意力機制
---
實作完整的自注意力機制，和上面大差不差，主要在於引入了**可以隨著model訓練的過程更新的`權重矩陣`($W_q$、$W_k$、$W_v$)**，這部分才確確實實的從原始輸入**X**中投影出獨立的**Q(Query)、K(Key)、V(Value)**三個向量，再丟進`Scaled Dot-Product Attention`公式計算

#### 3.4.1 逐步計算注意力權重

In [54]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [55]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [56]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

tensor([0.4306, 1.4551])


In [57]:
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [58]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


In [59]:
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [60]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [61]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


#### 3.4.2

In [62]:
import torch.nn as nn
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec

In [63]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [64]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec

In [65]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


### 3.5 使用因果注意力遮蔽未來的字詞

#### 3.5.1 應用因果注意力遮罩

In [66]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [67]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [68]:
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [69]:
row_sums = masked_simple.sum(dim=1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [70]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [71]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


#### 3.5.2 用dropout遮蔽額外的注意力權重

In [72]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6, 6)
print(dropout(example))

tensor([[2., 2., 2., 2., 2., 2.],
        [0., 2., 0., 0., 0., 0.],
        [0., 0., 2., 0., 2., 0.],
        [2., 2., 0., 0., 0., 2.],
        [2., 0., 0., 0., 0., 2.],
        [0., 2., 0., 0., 0., 0.]])


In [73]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.8966, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4921, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4350, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.0000, 0.0000, 0.0000, 0.0000]],
       grad_fn=<MulBackward0>)


#### 3.5.3 實作一個精簡的因果注意力機制

In [74]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [75]:
class CausalAttention(nn.Module):
    def __init__(self, d_in,d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

In [76]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])
